# Ejercicio: Extracción de datos de la API de TMDB (The Movie Database)

![imagen](https://www.themoviedb.org/assets/2/v4/logos/v2/blue_square_2-d537fb228cf3ded904ef09b136fe3fec72548ebc1fea3fbbd1ad9e36364db38b.svg)

Trabajaremos con **TMDB**, una de las bases de datos de cine más importantes del mundo. El objetivo es extraer información de películas para realizar un análisis de mercado inicial.

### 1. Registro y Documentación
Tendrás que [registrarte en TMDB](https://www.themoviedb.org/signup) para obtener tu **API Key (v3 auth)** y consultar la [documentación oficial](https://developer.themoviedb.org/reference/intro/getting-started).

### 2. Objetivo del Ejercicio
Queremos que consultes la API para que te devuelva la información de las películas que empiecen por la **inicial de tu nombre** (parámetro `query`). 

Debes guardar la información en un archivo `.csv` con la siguiente estructura de columnas:

| Columna | Descripción |
| :--- | :--- |
| **id** | ID interno de la película en TMDB |
| **title** | Título de la película |
| **release_date** | Fecha de estreno |
| **genres** | Nombres de los géneros (ej: "Acción, Comedia") |
| **vote_average** | Puntuación media de los usuarios |
| **overview** | Sinopsis o resumen de la trama |

### 3. El Reto: Mapeo de Géneros
A diferencia de otras APIs, el endpoint de búsqueda de películas devuelve los géneros como una lista de IDs numéricos (ej: `[28, 12]`). 

**Tu labor es:**
1. Consultar el endpoint de "Genre List" para obtener la relación entre IDs y nombres.
2. Sustituir los IDs en tu DataFrame final por los nombres reales de los géneros (separados por comas).

---

### Código de inicio
Aquí tienes el bloque para empezar a configurar tus llamadas:

```python
import requests
import pandas as pd

# Rellena estas variables
api_key = "TU_API_KEY_AQUÍ"
url_base = "[https://api.themoviedb.org/3](https://api.themoviedb.org/3)"
mi_inicial = "P" # Sustituye por tu inicial

In [ ]:
import requests
import pandas as pd

api_key = "9d7ecdec6d98ae084568cbf4cf95c3f8" 
url_base = "https://api.themoviedb.org/3"
mi_inicial = "E"

# 1. Obtener los nombres de los géneros
url_genres = f"{url_base}/genre/movie/list"
params_g = {'api_key': api_key, 'language': 'es-ES'}
res_g = requests.get(url_genres, params=params_g)

if res_g.status_code == 200:
    # Crear diccionario {id: nombre}
    mapa_generos = {g['id']: g['name'] for g in res_g.json()['genres']}
    
    # 2. Buscar películas por la inicial
    url_search = f"{url_base}/search/movie"
    params_s = {'api_key': api_key, 'query': mi_inicial, 'language': 'es-ES'}
    res_s = requests.get(url_search, params=params_s)
    
    if res_s.status_code == 200:
        peliculas = res_s.json().get('results', [])
        
        datos_lista = []
        for p in peliculas:
            # Mapeo de IDs a Nombres
            nombres_generos = [mapa_generos.get(i, "Desconocido") for i in p.get('genre_ids', [])]
            
            datos_lista.append({
                'id': p.get('id'),
                'title': p.get('title'),
                'release_date': p.get('release_date'),
                'genres': ", ".join(nombres_generos),
                'vote_average': p.get('vote_average'),
                'overview': p.get('overview')
            })
        
        # 3. Crear DataFrame y guardar CSV
        df = pd.DataFrame(datos_lista)
        df.to_csv('peliculas_tmdb.csv', index=False, encoding='utf-8-sig')
        print("¡Éxito! Archivo 'peliculas_tmdb.csv' creado correctamente.")
        print(df[['title', 'genres']].head())
    else:
        print(f"Error en búsqueda: {res_s.status_code}. Revisa si tu API Key está activa.")
else:
    print(f"Error en géneros: {res_g.status_code}. El servidor dice: {res_g.text}")




¡Éxito! Archivo 'peliculas_tmdb.csv' creado correctamente.
                                title                           genres
0                     Escuadrón letal  Acción, Crimen, Drama, Suspense
1  Peaky Blinders: El hombre inmortal                    Crimen, Drama
2               Shelter: El Protector         Acción, Crimen, Suspense
3                      Ruta de escape                 Crimen, Suspense
4            Asesinato en la Embajada       Misterio, Suspense, Acción


In [14]:
print(f"Total de películas procesadas con la inicial '{mi_inicial}': {len(df)}")


Total de películas procesadas con la inicial 'E': 20
